# Lab 21 — Fine-tuning LLMs · RUN ALL (T4)

Chay tu tren xuong. Runtime > Change runtime type > **T4 GPU** truoc khi bat dau.

| O | Lam gi | Thoi gian |
|---|---|---|
| 1 | clone + install | ~1 phut |
| 2 | smoke: import + unit test | ~30 giay |
| 3 | **core pipeline NB1 -> NB5** | ~80 phut |
| 4 | gatekeeper + in ket qua | ~10 giay |


In [ ]:
# @title 1. Setup — clone + install (chạy ô này trước)
import os, subprocess, sys

REPO = "https://github.com/PhanHieudc37/Day21-Track3-Finetuning-Lab-2A202601227-PhanVanHieu.git"
if not os.path.exists("Day21-Track3-Finetuning-Lab"):
    subprocess.run(["git", "clone", "-q", REPO], check=True)
os.chdir("/content/Day21-Track3-Finetuning-Lab")
subprocess.run(["git", "pull", "-q"], check=False)
sys.path.insert(0, "src")

# Install from requirements.txt, NOT a copied list. The copied list is how the
# torchao>=0.16 pin reached requirements.txt and this bootstrap on different days --
# and a bootstrap missing a pin does not fail here, it fails 10 minutes later inside
# get_peft_model(). One source of truth. torch is preinstalled on Colab and
# requirements.txt pins it compatibly, so that line is a no-op.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
               check=True)

import torch
print("commit :", subprocess.run(["git","rev-parse","--short","HEAD"],
                                 capture_output=True, text=True).stdout.strip())
print("GPU    :", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "NONE — Runtime > Change runtime type > T4 GPU")
if torch.cuda.is_available():
    print("VRAM   : %.1f GB" % (torch.cuda.get_device_properties(0).total_memory/1024**3))


In [ ]:
# @title 2. Smoke — imports, seed data, unit tests (no GPU needed)
!python scripts/verify.py --smoke


In [ ]:
# @title 3. Core pipeline — NB1 → NB5
# EVAL_LIMIT truncates both eval sets: "" = full run (submittable),
# 8 = ~fast smoke pass. STAGES lets you resume after a failure.
import os
COMPUTE_TIER = "T4"        # @param ["CPU","LAPTOP","T4","BIGGPU"]
EVAL_LIMIT   = ""          # @param ["", "4", "8", "16", "25"]
STAGES       = "nb1 nb2 nb3 nb4 nb5"   # @param {type:"string"}

os.environ["COMPUTE_TIER"] = COMPUTE_TIER
if EVAL_LIMIT:
    os.environ["EVAL_LIMIT"] = EVAL_LIMIT
else:
    os.environ.pop("EVAL_LIMIT", None)

from labkit import device
print(device.banner(), "\n")

!python scripts/colab_run.py {STAGES}


In [ ]:
# @title 4. Gatekeeper + results
!python scripts/verify.py
print("\n================ results/ ================")
!ls -la results/
!echo && echo "---- runs.csv ----" && cat results/runs.csv 2>/dev/null
!echo && echo "---- verdict.json ----" && cat results/verdict.json 2>/dev/null


## Bonus B1 + B4 (chỉ chạy sau khi core đã hoàn tất)

Hai ô dưới khôi phục artefact core, train thêm đúng các run còn thiếu, chạy NB6 và đóng gói kết quả. Thời gian dự kiến trên T4: khoảng 45–60 phút.

In [ ]:
# @title 5. Upload lab21_artifacts.zip đã tải từ lần chạy core
from google.colab import files
import pathlib, subprocess
uploaded = files.upload()
archives = [name for name in uploaded if name.lower().endswith('.zip')]
if len(archives) != 1:
    raise ValueError('Hãy upload đúng một file ZIP artefact')
subprocess.run(['unzip', '-o', archives[0], '-d', '.'], check=True)
required = pathlib.Path('adapters/correct/adapter_model.safetensors')
if not required.exists():
    raise FileNotFoundError(f'ZIP không có {required}')
print('Đã khôi phục adapter correct và results core.')


In [ ]:
# @title 6. Chạy B1 + B4 (T4, khoảng 45–60 phút)
import os, subprocess, sys
os.environ['COMPUTE_TIER'] = 'T4'
os.environ.pop('EVAL_LIMIT', None)
os.environ.pop('FORCE_RETRAIN', None)

# NB1 tái tạo data/split (artefact tải về không chứa split).
subprocess.run([sys.executable, 'notebooks/01_data_and_mask.py'], check=True)
# B1 cần adapter thứ hai để chứng minh hot-swap trên cùng base.
os.environ['ONLY'] = 'attn_only'
subprocess.run([sys.executable, 'notebooks/04_misconfig_autopsy.py'], check=True)
os.environ.pop('ONLY', None)
# B4 tái dùng correct r=16 và chỉ train thêm r=8, r=64.
subprocess.run([sys.executable, 'notebooks/07_bonus_rank_sweep.py'], check=True)
# B1: merge, assert không tụt quá 0.01, hot-swap correct + attn_only.
subprocess.run([sys.executable, 'notebooks/06_merge_and_serve.py'], check=True)
print('B1 + B4 đã chạy xong.')


In [ ]:
# @title 7. Verify và tải artefact bonus về máy
from google.colab import files
import pathlib, shutil, subprocess, sys
subprocess.run([sys.executable, 'scripts/verify.py'], check=True)
package = pathlib.Path('/content/lab21_bonus_artifacts')
if package.exists():
    shutil.rmtree(package)
shutil.copytree('results', package / 'results')
for name in ('correct', 'attn_only'):
    shutil.copytree(pathlib.Path('adapters') / name, package / 'adapters' / name)
archive = shutil.make_archive('/content/lab21_bonus_artifacts', 'zip', '/content/lab21_bonus_artifacts')
print('Đã tạo:', archive)
files.download(archive)
